# Query-based

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle')

if not Path('/content/data/train').exists():
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content/data
    !cd /content/data && unzip -q '*.zip' && rm *.zip
else:
    print('Data is here')
ROOT = Path('/content/data')

100% 9.88G/9.88G [08:53<00:00, 19.9MB/s]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PREP = Path('/content/prepared')
if not PREP.exists():
    !unzip -q /content/drive/MyDrive/УКУ/AI/hw2_prepared.zip -d /content
for split in ['train', 'val']:
    n = sum(1 for _ in (PREP / split / 'images').rglob('*.jpg'))
    print(f'{split}: {n} кадрів')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: 4868 кадрів
val: 571 кадрів


## Setup

In [ ]:
import random, cv2, numpy as np, torch
import torchvision.transforms.functional as TF

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
SEED, DEVICE = 42, 'cuda'

## Evaluation
Predictions are made on the original val frames (every 5th frame, 571 frames, not masked). Metrics are computed with pycocotools (`maxDets=500`); detections inside ignored regions are not penalised thanks to `iscrowd=1` in the ground truth. The same evaluation is used for every model.

In [ ]:
import json, contextlib, io, pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

with contextlib.redirect_stdout(io.StringIO()):
    gt = COCO(str(ROOT / 'val/annotations_coco.json'))
imgs_va = pd.DataFrame(gt.dataset['images']).rename(columns={'id': 'image_id'})
val_sub = imgs_va.sort_values(['video', 'frame_id'])
val_sub = val_sub[val_sub.groupby('video').cumcount() % 5 == 0]       # той самий кожен 5-й кадр
print('val frames for evaluation:', len(val_sub))

@torch.no_grad()
def predict(model, df, split='val', bs=4):
    """Передбачення на оригінальних кадрах -> список у COCO-форматі (bbox = [x, y, w, h] у пікселях)."""
    model.eval(); out = []
    for i in range(0, len(df), bs):
        rows = df.iloc[i:i + bs]
        ims = [TF.to_tensor(cv2.cvtColor(cv2.imread(str(ROOT / split / f)), cv2.COLOR_BGR2RGB)).to(DEVICE)
               for f in rows.file_name]
        with torch.autocast('cuda', dtype=torch.float16):
            res = model(ims)
        for iid, r in zip(rows.image_id, res):
            b = r['boxes'].float().cpu().numpy(); b[:, 2:] -= b[:, :2]      # xyxy -> xywh
            for box, s, l in zip(b, r['scores'].float().cpu().numpy(), r['labels'].cpu().numpy()):
                out.append({'image_id': int(iid), 'category_id': int(l),   # мітки 1..8 = COCO category_id
                            'bbox': [round(float(v), 2) for v in box], 'score': float(s)})
    return out

def evaluate(preds, img_ids):
    """COCO mAP + AP по класах."""
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, 'bbox')
        E.params.imgIds = list(img_ids); E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    res = {'mAP': s[0], 'AP50': s[1], 'AP75': s[2], 'AP_small': s[3], 'AP_medium': s[4], 'AP_large': s[5]}
    prec = E.eval['precision']                                           # [IoU, recall, class, area, maxDets]
    for k, c in enumerate(CLASSES):
        p = prec[:, :, k, 0, -1]
        res[f'AP_{c}'] = float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return res

val frames for evaluation: 571


In [ ]:
!pip -q install ultralytics

In [ ]:
import os, yaml
from ultralytics import RTDETR

# ---- budget knobs: keep IMGSZ equal to your CNN run for a fair comparison ----
EPOCHS, IMGSZ, BATCH = 6, 1024, 4            # lower BATCH or IMGSZ on out-of-memory
RUN_DIR, RUN_NAME = '/content/drive/MyDrive/hw2_runs', 'rtdetr_l_6ep'

# ---- data: PREP/{train,val}/images/**.jpg with YOLO labels in PREP/{train,val}/labels/**.txt ----
counts = {}
for split in ['train', 'val']:
    lbls = list((PREP / split / 'labels').rglob('*.txt'))
    assert lbls, f'No YOLO labels in {PREP / split / "labels"}: the prepared data is in another format'
    counts[split] = [sum(1 for line in open(p) if line.strip()) for p in lbls]
busiest = max(counts['train'] + counts['val'])
print(f'objects per frame: mean {np.mean(counts["train"]):.0f}, max {busiest}')

DATA_YAML = '/content/rtdetr_data.yaml'
yaml.safe_dump({'path': str(PREP), 'train': 'train/images', 'val': 'val/images',
                'names': dict(enumerate(CLASSES))}, open(DATA_YAML, 'w'))

# ---- queries: about 1.2x the busiest frame, rounded up to 100, kept within [300, 900] ----
NQ = int(min(900, max(300, 100 * np.ceil(1.2 * busiest / 100))))
print('decoder queries:', NQ)

def set_queries(trainer):
    """Queries come from encoder top-k, so the count can change without losing pretrained weights."""
    for m in (trainer.model, trainer.ema.ema):
        m.model[-1].num_queries = NQ

model = RTDETR('rtdetr-l.pt')                # COCO-pretrained; the class heads are re-initialised for 8 classes
model.add_callback('on_train_start', set_queries)

model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    nbs=16,                                  # accumulate gradients to an effective batch of 16
    optimizer='AdamW', lr0=1e-4, lrf=0.1, weight_decay=1e-4,
    warmup_epochs=1,                         # the default of 3 would be half of the run
    close_mosaic=2,                          # last 2 epochs without mosaic
    amp=True, seed=SEED, device=0, workers=os.cpu_count(),
    save_period=1,                           # one checkpoint per epoch for the mAP-over-training plot
    project=RUN_DIR, name=RUN_NAME, exist_ok=True,
)



Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
objects per frame: mean 45, max 158
decoder queries: 300
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rtdetr_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=Tr

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/6      6.89G       1.39     0.4822     0.2435        266       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.0it/s 5:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 6.8it/s 10.6s
                   all        571      22650      0.218      0.324      0.093     0.0449

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/6      7.39G     0.6487     0.7913    0.07476        210       1024: 0% ──────────── 0/1217  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/6      7.39G     0.7877     0.6785    0.09918        261       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.1it/s 5:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.2it/s 7.9s
                   all        571      22650      0.199      0.377      0.167     0.0876

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/6      7.39G     0.8327     0.6091    0.08178        266       1024: 0% ──────────── 0/1217  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (4 outputs left out)

        5/6      7.39G      0.598     0.5103    0.06533        242       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.1it/s 4:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.5it/s 7.6s
                   all        571      22650      0.572      0.435      0.444      0.226

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        6/6      7.39G     0.6647     0.5164    0.06495        235       1024: 0% ──────────── 0/1217  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        6/6      7.39G     0.5831     0.5007    0.06271        103       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.1it/s 4:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.3it/s 7.7s
                   all        571      22650      0.554      0.431      0.439      0.227

6 epochs completed in 0.518 hours.
Optimizer stripped from /content/drive/MyDrive/hw2_runs/rtdetr_l_6ep/weights/last.pt, 66.4MB
Optimizer stripped from /content/drive/MyDrive/hw2_runs/rtdetr_l_6ep/weights/best.pt, 66.4MB

Validating /content/drive/MyDrive/hw2_runs/rtdetr_l_6ep/weights/best.pt...
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.4it/s 7.6s
                   all        571      22650      0.54

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd

hist = pd.read_csv(f'{RUN_DIR}/{RUN_NAME}/results.csv')
hist.columns = hist.columns.str.strip()
print(hist[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']])

   epoch  metrics/mAP50(B)  metrics/mAP50-95(B)
0      1           0.09303              0.04486
1      2           0.16677              0.08763
2      3           0.37998              0.19768
3      4           0.43959              0.22736
4      5           0.44425              0.22642
5      6           0.43931              0.22749


In [ ]:
def predict_rtdetr(weights, df, split='val', imgsz=IMGSZ, bs=8):
    m, out = RTDETR(weights), []
    files = [str(ROOT / split / f) for f in df.file_name]
    for i in range(0, len(df), bs):
        res = m.predict(files[i:i + bs], imgsz=imgsz, conf=0.001, max_det=500, verbose=False)
        for iid, r in zip(df.image_id.iloc[i:i + bs], res):
            b = r.boxes.xyxy.cpu().numpy(); b[:, 2:] -= b[:, :2]            # xyxy -> xywh
            for box, s, l in zip(b, r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
                out.append({'image_id': int(iid), 'category_id': int(l) + 1,  # 0..7 -> 1..8
                            'bbox': [round(float(v), 2) for v in box], 'score': float(s)})
    return out

W = f'{RUN_DIR}/{RUN_NAME}/weights'
res_rtdetr = evaluate(predict_rtdetr(f'{W}/best.pt', val_sub), val_sub.image_id)
# per-epoch curve with your metric: loop over f'{W}/epoch{k}.pt' for k in range(EPOCHS)

In [ ]:
display(pd.Series(res_rtdetr))

,0
mAP,0.209002
AP50,0.412744
AP75,0.182885
AP_small,0.095963
AP_medium,0.185841
AP_large,0.388437
AP_person,0.245517
AP_bicycle,0.159597
AP_car,0.407291
AP_van,0.239151


Experiment 1: learning rates (backbone vs transformer, clean warm-up)
Change: all parameter groups warm up from zero, and the backbone (layers 0–9) trains at 1e-5 while the encoder and decoder stay at 1e-4.
Reference: The RT-DETR recipe pairs a 1e-5 backbone rate with a 1e-4 base rate and a linear warm-up from near zero. RT-DETRv2 keeps 1e-4 for the detector across sizes and assigns bigger backbones smaller rates, 1e-5 for its L model.
arxiv.org
arxiv.org
Expectation: high confidence of a gain, mostly from the warm-up fix. To attribute the gain between the two parts, add one run with backbone_mult=1.0.


In [ ]:
from ultralytics import RTDETR
from ultralytics.models.rtdetr.train import RTDETRTrainer


class TunedRTDETRTrainer(RTDETRTrainer):
    """RT-DETR trainer with a separate learning rate for the CNN backbone (layers listed under `backbone:`)."""
    backbone_mult = 1.0

    def build_optimizer(self, model, name='auto', lr=0.001, momentum=0.9, decay=1e-5, iterations=1e5):
        opt = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        assert isinstance(opt, torch.optim.AdamW), 'this trainer expects optimizer="AdamW"'
        n_bb = len(model.yaml['backbone'])                                # 10 layers = HGNetv2
        bb = {id(p) for layer in model.model[:n_bb] for p in layer.parameters()}
        groups = []
        for g in opt.param_groups:                                        # weight / bn / bias
            for part, mult in (('backbone', self.backbone_mult), ('transformer', 1.0)):
                params = [p for p in g['params'] if (id(p) in bb) == (part == 'backbone')]
                if params:
                    groups.append({'params': params, 'lr': g['lr'] * mult, 'weight_decay': g['weight_decay'],
                                   'param_group': g['param_group'], 'part': part})
        for g in groups:
            print(f"lr group {g['part']:11s} {g['param_group']:6s} tensors={len(g['params']):3d} lr={g['lr']:.1e}")
        return torch.optim.AdamW(groups, lr=lr, betas=(momentum, 0.999), weight_decay=0.0,
                                 fused=next(model.parameters()).is_cuda)


def train_rtdetr(name, backbone_mult=1.0, warmup_bias_lr=0.1, num_dn=100, **overrides):
    def set_denoising(trainer):
        for m in (trainer.model, trainer.ema.ema):
            m.model[-1].num_denoising = num_dn
        print('denoising budget:', trainer.model.model[-1].num_denoising)

    TunedRTDETRTrainer.backbone_mult = backbone_mult
    model = RTDETR('rtdetr-l.pt')
    model.add_callback('on_train_start', set_denoising)
    args = dict(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=16,
        optimizer='AdamW', lr0=1e-4, lrf=0.1, weight_decay=1e-4,
        warmup_epochs=1, warmup_bias_lr=warmup_bias_lr, close_mosaic=2,
        amp=True, seed=SEED, device=0, workers=os.cpu_count(), save_period=1,
        project=RUN_DIR, name=name, exist_ok=True)
    args.update(overrides)
    model.train(trainer=TunedRTDETRTrainer, **args)
    return f'{RUN_DIR}/{name}'


runs = {'baseline': f'{RUN_DIR}/rtdetr_l_6ep'}
runs['exp1_lr'] = train_rtdetr('rtdetr_exp1_lr', backbone_mult=0.1, warmup_bias_lr=0.0)
runs['exp2_lr_dn'] = train_rtdetr('rtdetr_exp2_lr_dn', backbone_mult=0.1, warmup_bias_lr=0.0, num_dn=500)

# same evaluate, same 571 frames
table = pd.DataFrame({k: evaluate(predict_rtdetr(f'{d}/weights/best.pt', val_sub), val_sub.image_id)
                      for k, d in runs.items()}).T
display(table[['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']])

for k, d in runs.items():
    h = pd.read_csv(f'{d}/results.csv'); h.columns = h.columns.str.strip()
    plt.plot(h['epoch'], h['metrics/mAP50-95(B)'], marker='o', label=k)
plt.xlabel('epoch'); plt.ylabel('val mAP50-95 (Ultralytics val)'); plt.legend(); plt.grid(alpha=.3); plt.show()

Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rtdetr_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=None, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_exp1_lr, nbs=16, nms=None, opset=None, opti

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/6      7.36G      1.406     0.4257     0.3532        266       1024: 100% ━━━━━━━━━━━━ 1217/1217 3.9it/s 5:13
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 8.2it/s 8.8s
                   all        571      22650      0.453      0.407      0.375        0.2

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/6      7.36G     0.7733     0.4843     0.1073        210       1024: 0% ──────────── 0/1217  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/6      7.36G     0.8373     0.4722     0.1155        261       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.0it/s 5:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.9it/s 7.3s
                   all        571      22650      0.504      0.462      0.463      0.238

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/6      7.36G     0.8634     0.4569       0.09        266       1024: 0% ──────────── 0/1217  0.2s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (15 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        5/6      15.6G       0.62     0.4787    0.07002        242       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.0it/s 5:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.8it/s 7.3s
                   all        571      22650      0.623      0.496      0.517      0.272

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        6/6      15.6G     0.6896     0.4883    0.08272        235       1024: 0% ──────────── 0/1217  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        6/6      15.6G     0.6093     0.4723    0.06809        103       1024: 100% ━━━━━━━━━━━━ 1217/1217 4.1it/s 5:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 9.8it/s 7.3s
                   all        571      22650      0.644      0.507      0.529      0.278

6 epochs completed in 0.530 hours.
Optimizer stripped from /content/drive/MyDrive/hw2_runs/rtdetr_exp2_lr_dn/weights/last.pt, 66.4MB
Optimizer stripped from /content/drive/MyDrive/hw2_runs/rtdetr_exp2_lr_dn/weights/best.pt, 66.4MB

Validating /content/drive/MyDrive/hw2_runs/rtdetr_exp2_lr_dn/weights/best.pt...
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 10.4it/s 7.0s
                   all        571     